In [11]:
from langchain_core.output_parsers import JsonOutputParser
from langchain.output_parsers import StructuredOutputParser, ResponseSchema, PydanticToolsParser, PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.messages import AIMessage

In [12]:
class MovieRecommendation(BaseModel):
    title: str = Field(description="The title of the movie")
    genre: str = Field(description="The primary genre of the movie")
    release_year: int = Field(description="The year the movie was released")

In [13]:
# ==========================================
# 1. JsonOutputParser Example
# ==========================================
def demonstrate_json_parser():
    print("--- 1. JsonOutputParser ---")
    parser = JsonOutputParser(pydantic_object=MovieRecommendation)
    
    format_instructions = parser.get_format_instructions()
    print("Format Instructions:\n", format_instructions)
    
    simulated_llm_output = '{"title": "Inception", "genre": "Sci-Fi", "release_year": 2010}'
    parsed_result = parser.parse(simulated_llm_output)
    
    print("Parsed Output:", parsed_result)
    print("Type:", type(parsed_result))
    print("-" * 40)

In [14]:
demonstrate_json_parser()

--- 1. JsonOutputParser ---
Format Instructions:
 STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not i

In [15]:
# ==========================================
# 2. StructuredOutputParser Example
# ==========================================
def demonstrate_structured_parser():
    print("--- 2. StructuredOutputParser ---")
    
    # Aligned fields mirroring MovieRecommendation
    response_schemas = [
        ResponseSchema(name="title", description="The title of the movie"),
        ResponseSchema(name="genre", description="The primary genre of the movie"),
        ResponseSchema(name="release_year", description="The year the movie was released (as a number)")
    ]
    
    parser = StructuredOutputParser.from_response_schemas(response_schemas)
    format_instructions = parser.get_format_instructions()
    print("Format Instructions:\n", format_instructions)
    
    simulated_llm_output = """```json
{
    "title": "Interstellar",
    "genre": "Sci-Fi",
    "release_year": 2014
}
```"""
    
    parsed_result = parser.parse(simulated_llm_output)
    print("Parsed Output:", parsed_result)
    print("Type:", type(parsed_result))
    print("-" * 40)

In [16]:
demonstrate_structured_parser()

--- 2. StructuredOutputParser ---
Format Instructions:
 The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"title": string  // The title of the movie
	"genre": string  // The primary genre of the movie
	"release_year": string  // The year the movie was released (as a number)
}
```
Parsed Output: {'title': 'Interstellar', 'genre': 'Sci-Fi', 'release_year': 2014}
Type: <class 'dict'>
----------------------------------------


In [17]:
# ==========================================
# 3. Tool Output Parser Example (PydanticToolsParser)
# ==========================================
def demonstrate_tool_output_parser():
    print("--- 3. Tool Output Parser (PydanticToolsParser) ---")
    
    # Pass the MovieRecommendation Pydantic class directly as the tool schema
    parser = PydanticToolsParser(tools=[MovieRecommendation])
    
    # Simulated AIMessage where the LLM invokes the 'MovieRecommendation' tool
    simulated_message = AIMessage(
        content="",
        tool_calls=[
            {
                "name": "MovieRecommendation",
                "args": {
                    "title": "The Matrix",
                    "genre": "Action / Sci-Fi",
                    "release_year": 1999
                },
                "id": "call_movie_rec_123"
            }
        ]
    )
    
    # Use .invoke() to parse the AIMessage into a MovieRecommendation instance
    parsed_result = parser.invoke(simulated_message)
    
    print("Parsed Tool Call Object:", parsed_result[0])
    print("Movie Title:", parsed_result[0].title)
    print("Movie Genre:", parsed_result[0].genre)
    print("Release Year:", parsed_result[0].release_year)
    print("-" * 40)

In [18]:
demonstrate_tool_output_parser()

--- 3. Tool Output Parser (PydanticToolsParser) ---
Parsed Tool Call Object: title='The Matrix' genre='Action / Sci-Fi' release_year=1999
Movie Title: The Matrix
Movie Genre: Action / Sci-Fi
Release Year: 1999
----------------------------------------


In [19]:
# ==========================================
# 4. PydanticOutputParser Example (Returns a Pydantic Object)
# ==========================================
def demonstrate_pydantic_output_parser():
    print("--- 3. PydanticOutputParser ---")
    
    parser = PydanticOutputParser(pydantic_object=MovieRecommendation)
    
    format_instructions = parser.get_format_instructions()
    print("Format Instructions:\n", format_instructions)
    
    simulated_llm_output = '{"title": "The Dark Knight", "genre": "Action", "release_year": 2008}'
    parsed_result = parser.parse(simulated_llm_output)
    
    print("Parsed Output:", parsed_result)
    print("Type:", type(parsed_result))
    print("Accessing attribute via dot notation:", parsed_result.title)
    print("-" * 40)

In [20]:
 demonstrate_pydantic_output_parser()

--- 3. PydanticOutputParser ---
Format Instructions:
 The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "The title of the movie", "title": "Title", "type": "string"}, "genre": {"description": "The primary genre of the movie", "title": "Genre", "type": "string"}, "release_year": {"description": "The year the movie was released", "title": "Release Year", "type": "integer"}}, "required": ["title", "genre", "release_year"]}
```
Parsed Output: title='The Dark Knight' genre='Action' release_year=2008
Type: <class '__main__.MovieRecommendation'>
Accessing att